In [ ]:
from __future__ import annotations

import importlib.util
import json
import random
import sys
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.transforms import functional as TF

ROOT = Path('.').resolve()
KAGGLE = Path('/kaggle').exists()

def _find_kaggle_dataset_root() -> Path:
    # Prefer explicitly known mount patterns first for speed/stability.
    direct_candidates = [
        Path('/kaggle/input/datasets/sinfilroamigo/mlds-a2/Dataset'),
        Path('/kaggle/input/mlds-a2/Dataset'),
    ]
    for c in direct_candidates:
        if (c / 'train').exists() and (c / 'test').exists():
            return c

    # Generic fallback: look for any Dataset folder containing train+test.
    for p in Path('/kaggle/input').rglob('Dataset'):
        if (p / 'train').exists() and (p / 'test').exists():
            return p

    raise FileNotFoundError(
        'Could not find Dataset/ under /kaggle/input. Please attach the dataset in Kaggle and rerun.'
    )

if KAGGLE:
    DATA_ROOT = _find_kaggle_dataset_root()
    WORK_ROOT = Path('/kaggle/working')
    SUBMISSION_ROOT = WORK_ROOT # / '25942_MAKWANA_JEVINKUMAR_Assignment_2'
else:
    local_candidates = [ROOT / 'data' / 'Dataset', ROOT / 'data', ROOT / 'Dataset', ROOT, ROOT.parent / 'Dataset', ROOT.parent]
    DATA_ROOT = next(
        (p for p in local_candidates if (p / 'train').exists() and (p / 'test').exists()),
        ROOT.parent,
    )
    SUBMISSION_ROOT = ROOT

CLASS_DIR = SUBMISSION_ROOT / 'classification'
SEG_DIR = SUBMISSION_ROOT / 'segmentation'
CLASS_WEIGHTS = CLASS_DIR / 'weights'
SEG_WEIGHTS = SEG_DIR / 'weights'

CLASS_DIR.mkdir(parents=True, exist_ok=True)
SEG_DIR.mkdir(parents=True, exist_ok=True)
CLASS_WEIGHTS.mkdir(parents=True, exist_ok=True)
SEG_WEIGHTS.mkdir(parents=True, exist_ok=True)

def _dedupe_paths(paths: List[Path]) -> List[Path]:
    out: List[Path] = []
    seen = set()
    for p in paths:
        key = str(p)
        if key in seen:
            continue
        seen.add(key)
        out.append(p)
    return out

def _build_pretrained_dirs() -> List[Path]:
    dirs: List[Path] = [
        ROOT / 'pretrained',
        ROOT / 'weights',
        Path.cwd() / 'pretrained',
        Path.cwd() / 'weights',
    ]

    if KAGGLE:
        dirs.extend([
            Path('/kaggle/working/pretrained'),
            Path('/kaggle/working/weights'),
            Path('/kaggle/working/MLDS/A2/A2-V4/pretrained'),
            Path('/kaggle/working/MLDS/A2/A2-V4/weights'),
            Path('/kaggle/working/MLDS/A2/A2-V3/pretrained'),
            Path('/kaggle/working/MLDS/A2/A2-V3/weights'),
        ])

        # Also include any mounted folder named pretrained under Kaggle input.
        input_root = Path('/kaggle/input')
        if input_root.exists():
            for p in input_root.rglob('pretrained'):
                dirs.append(p)

    return _dedupe_paths(dirs)

def _mkdir_if_writable(path: Path) -> None:
    # Avoid attempting writes under /kaggle/input (read-only).
    text = str(path)
    if KAGGLE and text.startswith('/kaggle/input'):
        return
    try:
        path.mkdir(parents=True, exist_ok=True)
    except Exception:
        pass

LOCAL_PRETRAINED_DIRS = _build_pretrained_dirs()
for _d in LOCAL_PRETRAINED_DIRS:
    _mkdir_if_writable(_d)

SEARCH_ROOTS = [ROOT, Path.cwd()]
if KAGGLE:
    SEARCH_ROOTS.extend([Path('/kaggle/working'), Path('/kaggle/input')])
SEARCH_ROOTS = _dedupe_paths(SEARCH_ROOTS)

def _extract_state_dict(state):
    if isinstance(state, dict):
        for key in ('state_dict', 'model_state', 'model', 'weights'):
            if key in state and isinstance(state[key], dict):
                return state[key]
    return state

def _strip_module_prefix_if_needed(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    if not state_dict:
        return state_dict
    keys = list(state_dict.keys())
    if all(k.startswith('module.') for k in keys):
        return {k[len('module.'):]: v for k, v in state_dict.items()}
    return state_dict

def _try_load_weights_from_path(model: nn.Module, ckpt_path: Path, tag: str, strict: bool) -> bool:
    try:
        raw_state = torch.load(ckpt_path, map_location='cpu')
        state = _extract_state_dict(raw_state)
        if not isinstance(state, dict):
            return False
        state = _strip_module_prefix_if_needed(state)
        model.load_state_dict(state, strict=strict)
        print(f'[{tag}] Loaded local pretrained weights from: {ckpt_path}', flush=True)
        return True
    except Exception as e:
        print(f'[{tag}] Found {ckpt_path} but failed to load ({e}). Trying next candidate.', flush=True)
        return False

def _load_local_torchvision_weights(
    model: nn.Module,
    candidate_filenames: List[str],
    tag: str,
    fallback_error: Exception,
    strict: bool = False,
    ) -> nn.Module:
    # Pass 1: direct lookup in known directories.
    for base_dir in LOCAL_PRETRAINED_DIRS:
        for name in candidate_filenames:
            ckpt_path = base_dir / name
            if ckpt_path.exists() and _try_load_weights_from_path(model, ckpt_path, tag, strict):
                return model

    # Pass 2: recursive search in major roots for exact file names.
    print(f'[{tag}] Direct lookup failed; trying recursive search for: {candidate_filenames}', flush=True)
    seen = set()
    for name in candidate_filenames:
        for root in SEARCH_ROOTS:
            if not root.exists():
                continue
            try:
                for found in root.rglob(name):
                    key = str(found)
                    if key in seen:
                        continue
                    seen.add(key)
                    if _try_load_weights_from_path(model, found, tag, strict):
                        return model
            except Exception:
                continue

    print(f'[{tag}] Could not use torchvision download ({fallback_error}); using random init.', flush=True)
    print(f'[{tag}] Checked dirs: {LOCAL_PRETRAINED_DIRS}', flush=True)
    return model

_ORIG_RESNET50 = models.resnet50
_ORIG_MOBILENET_V2 = models.mobilenet_v2

def _resnet50_offline_safe(*args, **kwargs):
    requested_weights = kwargs.get('weights', None)
    if requested_weights is None:
        return _ORIG_RESNET50(*args, **kwargs)
    try:
        return _ORIG_RESNET50(*args, **kwargs)
    except Exception as e:
        base_kwargs = dict(kwargs)
        base_kwargs['weights'] = None
        model = _ORIG_RESNET50(*args, **base_kwargs)
        candidates = [
            'resnet50-11ad3fa6.pth',
            'resnet50_imagenet1k_v2.pth',
            'resnet50_imagenet.pth',
            'resnet50.pth',
        ]
        return _load_local_torchvision_weights(model, candidates, 'CLS', e, strict=False)

def _mobilenet_v2_offline_safe(*args, **kwargs):
    requested_weights = kwargs.get('weights', None)
    if requested_weights is None:
        return _ORIG_MOBILENET_V2(*args, **kwargs)
    try:
        return _ORIG_MOBILENET_V2(*args, **kwargs)
    except Exception as e:
        base_kwargs = dict(kwargs)
        base_kwargs['weights'] = None
        model = _ORIG_MOBILENET_V2(*args, **base_kwargs)
        candidates = [
            'mobilenet_v2-7ebf99e0.pth',
            'mobilenet_v2_imagenet1k_v1.pth',
            'mobilenet_v2_imagenet.pth',
            'mobilenet_v2.pth',
        ]
        return _load_local_torchvision_weights(model, candidates, 'SEG', e, strict=False)

models.resnet50 = _resnet50_offline_safe
models.mobilenet_v2 = _mobilenet_v2_offline_safe

CLASSES:List[str] = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
NUM_CLASSES = 21

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('ROOT:', ROOT)
print('CWD:', Path.cwd())
print('KAGGLE mode:', KAGGLE)
print('DATA_ROOT (read):', DATA_ROOT)
print('SUBMISSION_ROOT (write):', SUBMISSION_ROOT)
print('Offline pretrained dirs:', LOCAL_PRETRAINED_DIRS)
print('Search roots:', SEARCH_ROOTS)
print('Using device:', device)

## Part 1: Principal Component Analysis (PCA) using NumPy

### Problem Statement & Objectives
- Implement Principal Component Analysis (PCA) **from scratch using the NumPy library** without high-level decomposition toolkits (`sklearn.decomposition` or `torch.linalg`).
- Handle high-dimensional natural image data:
  1. **Approach A (Global Image PCA):** Resizing images to a fixed $32 \times 32$ resolution ($D = 1024$) and decomposing the global image covariance.
  2. **Approach B (Patch-Based PCA):** Partitioning arbitrary-dimension images into local $p \times p$ patches ($p=12, d=144$), enabling dimension reduction without distorting native aspect ratios or requiring uniform image dimensions.
- Perform systematic experimentation across different component counts ($k$), evaluate cumulative explained variance and reconstruction Mean Squared Error (MSE), and visualize the results.

In [ ]:
# Step 1: Dataset Inspection & Summary
train_dir = DATA_ROOT / "train"
images_dir = train_dir / "images"
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

image_paths = sorted(
    p for p in images_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
)

print(f"Total training images found: {len(image_paths)}")

# Collect image dimensions
widths, heights, sizes = [], [], []
for p in image_paths[:100]:  # Sample first 100 for fast overview
    with Image.open(p) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)
    sizes.append(p.stat().st_size)

print(f"Resolution range: Width [{min(widths)}, {max(widths)}], Height [{min(heights)}, {max(heights)}]")
print(f"Average resolution: {int(np.mean(widths))} x {int(np.mean(heights))}")
print(f"Sample images inspected: {len(widths)}")


In [ ]:
# Approach A: Global Resized Image PCA using NumPy (SVD)
import matplotlib.pyplot as plt

GLOBAL_SIZE = 32
D_GLOBAL = GLOBAL_SIZE * GLOBAL_SIZE  # 1024 dimensions
N_SAMPLES = min(500, len(image_paths))

# 1. Load and flatten resized images
rows = []
for p in image_paths[:N_SAMPLES]:
    with Image.open(p) as img:
        arr = np.asarray(img.convert('L').resize((GLOBAL_SIZE, GLOBAL_SIZE), Image.Resampling.BILINEAR), dtype=np.float32) / 255.0
        rows.append(arr.flatten())

X_global = np.stack(rows, axis=0)  # Shape: (N, 1024) <- (N, D)
print(f"Global PCA input matrix shape: {X_global.shape} (N={X_global.shape[0]}, D={X_global.shape[1]})")

# 2. Mean centering
mean_global = np.mean(X_global, axis=0, keepdims=True)
X_centered = X_global - mean_global

# 3. Compact SVD: X_centered = U @ diag(S) @ Vt
U_g, S_g, Vt_g = np.linalg.svd(X_centered, full_matrices=False)
print(f"U shape:  {U_g.shape}   <-- Left singular vectors")
print(f"S shape:  {S_g.shape}    <-- Singular values")
print(f"Vt shape: {Vt_g.shape}   <-- Right singular vectors")

eigvals_global = (S_g ** 2) / (X_global.shape[0] - 1)
var_ratio_global = eigvals_global / np.sum(eigvals_global)
cum_var_global = np.cumsum(var_ratio_global)

# 4. Experimentation across candidate k values
k_candidates_global = [8, 16, 32, 64, 128]
global_results = []

for k in k_candidates_global:
    basis = Vt_g[:k]  # (k, D)
    # Project: Z = (X - mean) @ basis.T
    Z = X_centered @ basis.T
    # Reconstruct: X_hat = Z @ basis + mean
    X_recon = Z @ basis + mean_global
    mse = float(np.mean((X_global - X_recon) ** 2))
    comp_ratio = D_GLOBAL / k
    global_results.append({
        'k_components': k,
        'compression_factor': f"{comp_ratio:.1f}x",
        'explained_variance_%': f"{cum_var_global[k - 1] * 100:.2f}%",
        'reconstruction_mse': f"{mse:.6f}"
    })

df_global = pd.DataFrame(global_results)
print("\n--- Global Resized Image PCA Results ---")
print(df_global)

# 5. Visual comparison for Global PCA
fig, axes = plt.subplots(2, 4, figsize=(11, 5))
fig.suptitle(f"Global PCA (32x32) Reconstruction at k=64 (MSE: {df_global.loc[3, 'reconstruction_mse']})", fontsize=12)

sample_k = 64
basis_sample = Vt_g[:sample_k]
X_recon_sample = (X_centered[:4] @ basis_sample.T) @ basis_sample + mean_global

for i in range(4):
    axes[0, i].imshow(X_global[i].reshape(GLOBAL_SIZE, GLOBAL_SIZE), cmap='gray')
    axes[0, i].set_title(f"Original #{i+1}")
    axes[0, i].axis('off')
    
    axes[1, i].imshow(np.clip(X_recon_sample[i].reshape(GLOBAL_SIZE, GLOBAL_SIZE), 0.0, 1.0), cmap='gray')
    axes[1, i].set_title(f"Recon (k={sample_k})")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Approach A (Alternative): Column-Oriented SVD Formulation
# In math textbooks, data points are often arranged as columns: A in R^{D x N}
# Here, the principal directions live directly in the left singular vectors: basis = U[:, :k]

# 1. Arrange data as Column Matrix: A has shape (D, N) = (1024, 500)
A_col = X_global.T  # (D, N) = (1024, 500)
D_dim, N_samples = A_col.shape
print(f"Column data matrix shape: {A_col.shape} (D={D_dim} features, N={N_samples} images)")

# 2. Mean centering along columns (across the N images)
mu_col = np.mean(A_col, axis=1, keepdims=True)  # (D, 1) = (1024, 1)
A_centered = A_col - mu_col                      # (D, N) = (1024, 500)

# 3. Compact SVD: A_centered = U @ diag(S) @ Vt
U_col, S_col, Vt_col = np.linalg.svd(A_centered, full_matrices=False)

print(f"U shape:  {U_col.shape}   <-- Columns are eigenvectors in feature space R^D")
print(f"S shape:  {S_col.shape}    <-- Singular values")
print(f"Vt shape: {Vt_col.shape}   <-- Rows are in sample space R^N")

# Variance explained: divisor is N - 1 (A_col.shape[1] - 1)
eigvals_col = (S_col ** 2) / (A_col.shape[1] - 1)
var_ratio_col = eigvals_col / np.sum(eigvals_col)
cum_var_col = np.cumsum(var_ratio_col)

# 4. Experimentation across candidate k values (Column-Oriented)
k_candidates_col = [8, 16, 32, 64, 128]
col_results = []

for k in k_candidates_col:
    # In Column format: basis is the first k columns of 'U'!
    basis = U_col[:, :k]  # Shape: (D, k) = (1024, k)
    
    # Project: Z = basis^T @ (A - mu) -> shape: (k, N) = (k, 500)
    Z = basis.T @ A_centered
    
    # Reconstruct: A_hat = basis @ Z + mu -> shape: (D, N) = (1024, 500)
    A_recon = basis @ Z + mu_col
    
    # MSE between (D, N) matrices
    mse = float(np.mean((A_col - A_recon) ** 2))
    comp_ratio = D_GLOBAL / k
    col_results.append({
        'k_components': k,
        'compression_factor': f"{comp_ratio:.1f}x",
        'explained_variance_%': f"{cum_var_col[k - 1] * 100:.2f}%",
        'reconstruction_mse': f"{mse:.6f}"
    })

df_col = pd.DataFrame(col_results)
print("\n--- Global Resized Image PCA Results (Column-Oriented) ---")
print(df_col)

# 5. Visual comparison for Column-Oriented Global PCA
fig, axes = plt.subplots(2, 4, figsize=(11, 5))
fig.suptitle(f"Global PCA (32x32) Column-Oriented Reconstruction at k=64 (MSE: {df_col.loc[3, 'reconstruction_mse']})", fontsize=12)

sample_k = 64
basis_sample = U_col[:, :sample_k]  # (1024, 64)
# Reconstruct first 4 image columns: A_centered[:, :4] has shape (1024, 4)
A_recon_sample = basis_sample @ (basis_sample.T @ A_centered[:, :4]) + mu_col  # (1024, 4)

for i in range(4):
    # Column i is image i: A_col[:, i]
    axes[0, i].imshow(A_col[:, i].reshape(GLOBAL_SIZE, GLOBAL_SIZE), cmap='gray')
    axes[0, i].set_title(f"Original #{i+1}")
    axes[0, i].axis('off')
    
    axes[1, i].imshow(np.clip(A_recon_sample[:, i].reshape(GLOBAL_SIZE, GLOBAL_SIZE), 0.0, 1.0), cmap='gray')
    axes[1, i].set_title(f"Recon (k={sample_k})")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

### Approach B: Patch-Based PCA for Arbitrary Dimension Images (Pure NumPy)

In natural images, resizing every image to a small fixed square ($32 \times 32$) distorts aspect ratios and discards fine-grained local textures. 

**Patch-Based PCA** solves this by:
1. Decomposing full-resolution images of **any arbitrary dimension** into non-overlapping local patches of size $p \times p$ ($p=12$, feature dimension $d = 144$).
2. Learning a compact local basis via covariance eigendecomposition using **pure NumPy** (`np.linalg.eigh`).
3. Compressing and reconstructing the patches independently, and reassembling the full 2D image without resizing distortions.

In [ ]:
# Approach B Implementation: Pure NumPy PatchPCA
class PatchPCANumPy:
    """
    Patch-Based Principal Component Analysis implemented purely with NumPy.
    Extracts non-overlapping p x p patches, centers them, computes the covariance matrix,
    and solves the eigendecomposition using np.linalg.eigh.
    """
    def __init__(self, patch_size: int = 12):
        self.p = patch_size
        self.d = patch_size * patch_size  # Feature dimensionality (144 for p=12)
        self.mean_patch = None
        self.eigvals = None
        self.eigvecs = None
        self.explained_variance_ratio = None
        self.cum_explained_variance = None

    def _extract_patches(self, img_array: np.ndarray) -> Tuple[np.ndarray, Tuple[int, int]]:
        h, w = img_array.shape
        p = self.p
        h_crop = (h // p) * p
        w_crop = (w // p) * p
        cropped = img_array[:h_crop, :w_crop]

        patches = []
        for r in range(0, h_crop, p):
            for c in range(0, w_crop, p):
                patch = cropped[r : r + p, c : c + p].astype(np.float32)
                patches.append(patch.flatten())
        
        # Returns shape: (d, num_patches)
        return np.array(patches, dtype=np.float32).T, (h_crop, w_crop)

    def fit_from_paths(self, paths: List[Path], max_patches: int = 40000):
        patch_list = []
        total_collected = 0

        for pth in paths:
            with Image.open(pth) as img:
                arr = np.asarray(img.convert('L'), dtype=np.float32) / 255.0
            
            p_mat, _ = self._extract_patches(arr)
            patch_list.append(p_mat)
            total_collected += p_mat.shape[1]
            if total_collected >= max_patches:
                break

        # Shape: (d, N) where d = 144
        Y_all = np.hstack(patch_list)[:, :max_patches]
        print(f"Extracted {Y_all.shape[1]} patches of size {self.p}x{self.p} (Input dimension d={self.d})")

        # 1. Compute empirical mean patch and center data
        self.mean_patch = np.mean(Y_all, axis=1, keepdims=True)
        Y_centered = Y_all - self.mean_patch

        # 2. Compute sample covariance matrix: Cov = (Y_c @ Y_c.T) / (N - 1) -> (d, d)
        cov = (Y_centered @ Y_centered.T) / (Y_all.shape[1] - 1)

        # 3. Eigendecomposition with pure NumPy (np.linalg.eigh for symmetric matrices)
        eigenvalues, eigenvectors = np.linalg.eigh(cov)

        # eigh returns ascending order; sort in descending order of explained variance
        idx = np.argsort(eigenvalues)[::-1]
        self.eigvals = eigenvalues[idx]
        self.eigvecs = eigenvectors[:, idx]

        # 4. Explained variance ratios
        self.explained_variance_ratio = self.eigvals / np.sum(self.eigvals)
        self.cum_explained_variance = np.cumsum(self.explained_variance_ratio)
        print(f"Successfully computed PCA subspace using pure NumPy. Top eigenvalue: {self.eigvals[0]:.4f}")

    def transform_and_reconstruct(self, img_array: np.ndarray, k: int) -> Tuple[np.ndarray, float, float]:
        patches, (h_crop, w_crop) = self._extract_patches(img_array)
        p = self.p

        # Basis of top-k eigenvectors: shape (d, k)
        U_k = self.eigvecs[:, :k]

        # Project: c = U_k^T (y - y_mean) -> shape (k, num_patches)
        patches_centered = patches - self.mean_patch
        c = U_k.T @ patches_centered

        # Reconstruct: y_hat = y_mean + U_k @ c -> shape (d, num_patches)
        recon_patches = self.mean_patch + (U_k @ c)
        recon_patches = np.clip(recon_patches, 0.0, 1.0)

        # Assemble full 2D reconstructed image
        recon_img = np.zeros((h_crop, w_crop), dtype=np.float32)
        idx = 0
        for r in range(0, h_crop, p):
            for c_idx in range(0, w_crop, p):
                recon_img[r : r + p, c_idx : c_idx + p] = recon_patches[:, idx].reshape(p, p)
                idx += 1

        cropped_orig = img_array[:h_crop, :w_crop]
        mse = float(np.mean((cropped_orig - recon_img) ** 2))
        compression_ratio = self.d / k
        return recon_img, compression_ratio, mse

# Fit Patch PCA model
patch_pca = PatchPCANumPy(patch_size=12) # p=12, dim of each patch = 12 X 12 = 144 (flattened, (144,))
patch_pca.fit_from_paths(image_paths, max_patches=40000)


In [ ]:
# Experimentation with Different Number of Components (k) on Patch PCA
k_candidates = [2, 4, 8, 16, 32, 64]

sample_test_path = image_paths[0]
with Image.open(sample_test_path) as img:
    sample_img = np.asarray(img.convert('L'), dtype=np.float32) / 255.0

patch_results = []
reconstructions = {}

for k in k_candidates:
    recon, comp_factor, mse = patch_pca.transform_and_reconstruct(sample_img, k=k)
    reconstructions[k] = (recon, mse)
    patch_results.append({
        'k_components': k,
        'compression_factor': f"{comp_factor:.1f}x reduction",
        'dim_reduced': f"{patch_pca.d}D -> {k}D",
        'cum_explained_variance_%': f"{patch_pca.cum_explained_variance[k - 1] * 100:.2f}%",
        'sample_reconstruction_mse': f"{mse:.6f}"
    })

df_patch = pd.DataFrame(patch_results)
print(f"Test Image: {sample_test_path.name} (Resolution: {sample_img.shape[1]}x{sample_img.shape[0]})")
print("\n--- Patch-Based PCA Component Evaluation Table ---")
print(df_patch)


In [ ]:
# Visual Analysis: Scree Plot, Cumulative Variance, and Multi-k Reconstructions
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# 1. Eigenvalue spectrum (Scree plot)
axes[0].plot(range(1, 31), patch_pca.eigvals[:30], marker='o', markersize=4, color='#1f77b4', linewidth=1.8)
axes[0].set_title('Scree Plot (First 30 Eigenvalues)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Principal Component Index', fontsize=10)
axes[0].set_ylabel('Eigenvalue (Variance)', fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.5)

# 2. Cumulative Explained Variance
axes[1].plot(range(1, patch_pca.d + 1), patch_pca.cum_explained_variance * 100, color='#2ca02c', linewidth=2.0)
axes[1].axhline(y=90, color='orange', linestyle='--', label='90% Threshold')
axes[1].axhline(y=95, color='red', linestyle='--', label='95% Threshold')
axes[1].axvline(x=16, color='blue', linestyle=':', label='Selected k=16 (96.76%)')
axes[1].set_title('Cumulative Explained Variance Curve', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Components (k)', fontsize=10)
axes[1].set_ylabel('Cumulative Variance (%)', fontsize=10)
axes[1].legend(loc='lower right')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# 3. Visual Reconstruction Comparison across different k values
show_ks = [2, 8, 16, 64]
fig, axes = plt.subplots(1, len(show_ks) + 1, figsize=(16, 4))

h_crop = (sample_img.shape[0] // patch_pca.p) * patch_pca.p
w_crop = (sample_img.shape[1] // patch_pca.p) * patch_pca.p
axes[0].imshow(sample_img[:h_crop, :w_crop], cmap='gray')
axes[0].set_title(f"Original\n({w_crop}x{h_crop})", fontsize=10, fontweight='bold')
axes[0].axis('off')

for i, k in enumerate(show_ks):
    recon, mse = reconstructions[k]
    axes[i + 1].imshow(recon, cmap='gray')
    axes[i + 1].set_title(f"k={k} (MSE: {mse:.5f})\n{patch_pca.d / k:.1f}x compression", fontsize=10)
    axes[i + 1].axis('off')

plt.suptitle(f"Reconstruction Quality across Components on {sample_test_path.name}", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# 4. Top Learned Eigenpatches (Principal Basis Vectors)
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i in range(16):
    ax = axes[i // 8, i % 8]
    # Reshape the i-th eigenvector to p x p patch
    filter_img = patch_pca.eigvecs[:, i].reshape(patch_pca.p, patch_pca.p)
    ax.imshow(filter_img, cmap='coolwarm')
    ax.set_title(f"PC {i+1}", fontsize=8)
    ax.axis('off')

plt.suptitle("Top 16 Learned Eigenpatches (Spatial Feature Detectors: DC, Edges, Textures)", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


### Results & Observations for Part 1 (PCA)

1. **Eigenvalue Spectrum & Energy Compaction:**
   - The scree plot shows sharp eigenvalue decay: the first component alone accounts for the bulk of local patch energy (average luminance / DC component), capturing over $88\%$ of the total variance.
   - By component $k = 4$, cumulative variance reaches **$93.49\%$**, and by $k = 16$, it captures **$96.76\%$** of total variance.

2. **Reconstruction Fidelity(closeness) vs. Dimensionality Reduction Trade-off:**
   - At $k = 16$, the dimensionality is reduced from $144$ down to $16$ (a **$9.0\times$ compression factor**), achieving a low reconstruction MSE of $0.000925$. Perceptually, high-contrast edges and structural boundaries are faithfully preserved.
   - When $k = 2$ ($72\times$ compression), the model captures global shading but produces blurred textures. At $k = 64$ ($2.2\times$ compression), reconstruction is nearly indistinguishable from the original image (MSE: $0.000270$).
   - Therefore, **$k = 16$ offers an optimal trade-off** between dramatic compression ($88.9\%$ storage reduction) and visual fidelity ($>96.7\%$ variance preserved).

3. **Interpretability of Learned Eigenpatches:**
   - Visualizing the top eigenvectors as $12 \times 12$ spatial patches reveals that PCA naturally learns classical image primitives:
     - **PC 1:** Uniform DC illumination (constant intensity offset).
     - **PC 2 & PC 3:** Horizontal and vertical first-order edge gradients.
     - **PC 4 to PC 8:** Diagonal edges, saddle points, and corner gradients.
     - **Higher PCs:** High-frequency texture and localized noise variations.

4. **Global Image PCA vs. Patch-Based PCA:**
   - **Global PCA** ($32 \times 32$ images): Requires rigid downsampling and spatial alignment; captures global scene structure and lighting but sacrifices resolution.
   - **Patch PCA**: Operates on native high-resolution images of variable sizes without aspect-ratio distortion, effectively learning a compact local dictionary applicable to any natural image.